# Fine-tuning LLaMA 3.1 8B Base s QLoRA

Trénink LoRA adapteru pro `meta-llama/Meta-Llama-3.1-8B` (base) na datasetu 278 anotovaných českých titulků.

Prostředí: Google Colab Pro, GPU L4 (22 GB VRAM).

## 1. Instalace knihoven

In [1]:
%%capture
# Instalace unsloth + závislosti
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets transformers wandb

## 2. Připojení Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
PROJECT_ROOT = '/content/drive/MyDrive/bp-misleading-cs'

TRAIN_DATA_PATH = f'{PROJECT_ROOT}/data/processed/train_instruction.jsonl'
VAL_DATA_PATH = f'{PROJECT_ROOT}/data/processed/val_instruction.jsonl'
ADAPTER_OUTPUT_DIR = f'{PROJECT_ROOT}/models/llama_base_qlora_v1'

import os
os.makedirs(ADAPTER_OUTPUT_DIR, exist_ok=True)

assert os.path.exists(TRAIN_DATA_PATH), f'Chybí: {TRAIN_DATA_PATH}'
assert os.path.exists(VAL_DATA_PATH), f'Chybí: {VAL_DATA_PATH}'
print(f'train: {TRAIN_DATA_PATH}')
print(f'val:   {VAL_DATA_PATH}')
print(f'output: {ADAPTER_OUTPUT_DIR}')

✅ train: /content/drive/MyDrive/bp-misleading-cs/data/processed/train_instruction.jsonl
✅ val:   /content/drive/MyDrive/bp-misleading-cs/data/processed/val_instruction.jsonl
✅ output: /content/drive/MyDrive/bp-misleading-cs/models/llama_base_qlora_v1


## 3. Login do HuggingFace

In [4]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('✅ Přihlášeno do HuggingFace')

✅ Přihlášeno do HuggingFace


## 4. Login do Weights & Biases

In [5]:
import wandb

try:
    wandb_token = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_token)
    USE_WANDB = True
    print('✅ Přihlášeno do W&B')
except Exception as e:
    USE_WANDB = False
    print(f'⚠️ W&B nedostupné ({e}) — průběh tréninku se zaloguje jen v notebooku')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: o_zemanek (o_zemanek-univerzita-tom-e-bati-ve-zl-n-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Přihlášeno do W&B


## 5. Vypnutí Unsloth telemetrie

In [6]:
import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ['HF_HUB_OFFLINE'] = '0'

## 6. Stažení modelu LLaMA 3.1 8B Base (4-bit)

In [7]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Meta-Llama-3.1-8B',  # base verze, ne Instruct!
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=hf_token,
)

print(f'✅ Model načten: {model.config._name_or_path}')
print(f'   Parametrů: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model načten: unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit
   Parametrů: 4.6B


## 7. Aplikace LoRA adapteru

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'LoRA aplikováno')
print(f'Trénovatelných parametrů: {trainable / 1e6:.1f}M ({trainable / total * 100:.2f}%)')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA aplikováno
   Trénovatelných parametrů: 41.9M (0.90% z celku)


## 8. Načtení a formátování tréninkových dat

In [9]:
from datasets import load_dataset

train_dataset = load_dataset('json', data_files=TRAIN_DATA_PATH, split='train')
val_dataset = load_dataset('json', data_files=VAL_DATA_PATH, split='train')

print(f'Train: {len(train_dataset)} záznamů')
print(f'Val:   {len(val_dataset)} záznamů')
print(f'\nUkázka prvního záznamu:')
print(f'  instruction (prvních 100 znaků): {train_dataset[0]["instruction"][:100]}...')
print(f'  input:  {train_dataset[0]["input"]}')
print(f'  output: {train_dataset[0]["output"][:200]}...')

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 278 záznamů
Val:   40 záznamů

Ukázka prvního záznamu:
  instruction (prvních 100 znaků): Jsi expert na analýzu zpravodajských sdělení. Tvým úkolem je vyhodnotit česky psaný titulek a posoud...
  input:  Titulek: "Trumpova celní válka se světem dopadne i na Američany: Zdraží i jejich ikonické zboží!"
  output: {"A1-scope_of_missing_context": "HIGH", "A2-scope_of_missing_context_type": ["CAUSALITY", "SCOPE", "TIMEFRAME", "STATISTICAL_BASELINE"], "A3-quantification_present": "NONE", "B1-framing_present": "YES...


In [10]:
ALPACA_TEMPLATE = '''### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}'''

EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    text = ALPACA_TEMPLATE.format(
        instruction=example['instruction'],
        input=example['input'],
        output=example['output'],
    ) + EOS_TOKEN
    return {'text': text}

train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

print('=== UKÁZKA ===')
print(train_dataset[0]['text'][:500])
print('... (zkráceno) ...')
print(train_dataset[0]['text'][-300:])

Map:   0%|          | 0/278 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

=== UKÁZKA ===
### Instruction:
Jsi expert na analýzu zpravodajských sdělení. Tvým úkolem je vyhodnotit česky psaný titulek a posoudit jeho zavádějícnost podle definovaného schématu.

Vyplníš pole v pořadí A → B → C → D → E. Až po vyhodnocení bloků A-D rozhodneš o E1.

Vrať pouze validní JSON s následujícími poli (žádný text před ani po):
- A1-scope_of_missing_context: NONE / LOW / HIGH
- A2-scope_of_missing_context_type: array of MOTIVATION / CAUSALITY / SCOPE / TIMEFRAME / PROCESS_STAGE / UNCERTAINTY_LEVEL /
... (zkráceno) ...
["EMOTIONAL_LANGUAGE", "ABSOLUTE_CLAIMS", "CAUSAL_SHORTCUT", "CLICKBAIT_STYLE"], "B3-attribution_clarity": "MISSING", "C1-misinterpretation_risk": "MEDIUM", "C3-conspiracy_pattern": "NONE", "D1-sensitive_domain": "FINANCE", "E1-misleading_header_model_final": "POTENTIALLY_MISLEADING"}<|end_of_text|>


## 9. Trénink

In [15]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

if USE_WANDB:
    wandb.init(
        project='bp-misleading-cs',
        name='llama_base_qlora_v1',
        config={
            'model': 'Meta-Llama-3.1-8B',
            'dataset_size': len(train_dataset),
            'lora_rank': 16,
            'lora_alpha': 32,
            'learning_rate': 2e-4,
            'epochs': 3,
        }
    )

# Tokenizace dat
def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding='max_length',
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training args
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    output_dir=f'{PROJECT_ROOT}/checkpoints/llama_base_qlora_v1',
    eval_strategy='steps',
    eval_steps=20,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='wandb' if USE_WANDB else 'none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

trainer_stats = trainer.train()

Map:   0%|          | 0/278 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 278 | Num Epochs = 3 | Total steps = 105
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
20,0.190495,0.179397
40,0.137172,0.154833
60,0.131972,0.154004
80,0.115306,0.153813
100,0.115535,0.154727
105,0.110251,0.154812


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

In [16]:
print(f'Trénink dokončen')
print(f'Celkem kroků: {trainer_stats.global_step}')
print(f'Doba: {trainer_stats.metrics["train_runtime"]:.1f} s')
print(f'Final train loss: {trainer_stats.metrics["train_loss"]:.4f}')

✅ Trénink dokončen
   Celkem kroků: 105
   Doba: 1261.2 s
   Final train loss: 0.2747


## 10. Uložení adapteru

In [17]:
model.save_pretrained(ADAPTER_OUTPUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)

for f in os.listdir(ADAPTER_OUTPUT_DIR):
    full_path = os.path.join(ADAPTER_OUTPUT_DIR, f)
    size_mb = os.path.getsize(full_path) / 1024 / 1024
    print(f'  {f}: {size_mb:.1f} MB')

print(f'\nAdapter uložen do: {ADAPTER_OUTPUT_DIR}')

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/bp-misleading-cs/models/llama_base_qlora_v1/tokenizer_config.json.


  README.md: 0.0 MB
  adapter_model.safetensors: 160.1 MB
  adapter_config.json: 0.0 MB
  tokenizer_config.json: 0.0 MB
  tokenizer.json: 16.4 MB

✅ Adapter uložen do: /content/drive/MyDrive/bp-misleading-cs/models/llama_base_qlora_v1
   Můžeš ho stáhnout přes Drive UI nebo gdrive CLI.


## 11. Testovací inference na validačních datech

In [18]:
FastLanguageModel.for_inference(model)

def generate_response(instruction, input_text):
    prompt = ALPACA_TEMPLATE.format(
        instruction=instruction,
        input=input_text,
        output='',
    )
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.0,
        do_sample=False,
        use_cache=True,
    )

    full_output = tokenizer.batch_decode(outputs)[0]
    response = full_output.split('### Response:\n')[1].split(EOS_TOKEN)[0].strip()
    return response

for i in range(2):
    sample = val_dataset[i]
    print(f'=== Test {i+1} ===')
    print(f'INPUT: {sample["input"]}')
    print(f'\nGOLD ANSWER:\n{sample["output"]}')

    generated = generate_response(sample['instruction'], sample['input'])
    print(f'\nMODEL ANSWER:\n{generated}')
    print('\n' + '='*60 + '\n')

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test 1 ===
INPUT: Titulek: "První potvrzený pár StarDance Tour: Tatiana Dyková a Petr Čadek!"

GOLD ANSWER:
{"A1-scope_of_missing_context": "NONE", "A2-scope_of_missing_context_type": [], "A3-quantification_present": "NONE", "B1-framing_present": "NO", "B2-framing_type": [], "B3-attribution_clarity": "CLEAR", "C1-misinterpretation_risk": "LOW", "C3-conspiracy_pattern": "NONE", "D1-sensitive_domain": "NONE", "E1-misleading_header_model_final": "NOT_MISLEADING"}


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


MODEL ANSWER:
{"A1-scope_of_missing_context": "NONE", "A2-scope_of_missing_context_type": [], "A3-quantification_present": "NONE", "B1-framing_present": "NO", "B2-framing_type": [], "B3-attribution_clarity": "CLEAR", "C1-misinterpretation_risk": "LOW", "C3-conspiracy_pattern": "NONE", "D1-sensitive_domain": "NONE", "E1-misleading_header_model_final": "NOT_MISLEADING"}


=== Test 2 ===
INPUT: Titulek: "Trump nařídil vyplatit mzdy všem zaměstnancům ministerstva vnitřní bezpečnosti"

GOLD ANSWER:
{"A1-scope_of_missing_context": "LOW", "A2-scope_of_missing_context_type": ["CAUSALITY", "TIMEFRAME"], "A3-quantification_present": "NONE", "B1-framing_present": "NO", "B2-framing_type": [], "B3-attribution_clarity": "CLEAR", "C1-misinterpretation_risk": "LOW", "C3-conspiracy_pattern": "NONE", "D1-sensitive_domain": "POLITICS", "E1-misleading_header_model_final": "NOT_MISLEADING"}

MODEL ANSWER:
{"A1-scope_of_missing_context": "LOW", "A2-scope_of_missing_context_type": ["SCOPE", "TIMEFRAME"], "A

## 12. Ukončení W&B

In [19]:
if USE_WANDB:
    wandb.finish()
    print('W&B run ukončen')

eval/loss,█▁▁▁▁▁
eval/runtime,█▂▂▁▃▁
eval/samples_per_second,▁▇▇█▆█
eval/steps_per_second,▁▆██▆█
train/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇█████
train/global_step,▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇█████
train/grad_norm,▃▅▄▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▄▇███▇▇▇▆▅▅▄▄▃▃▂▂▁▁▁▁
train/loss,█▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.15481
eval/runtime,17.4149


✅ W&B run ukončen — graf si stáhneš z https://wandb.ai/
